In [1]:
import pandas as pd
import requests
from io import StringIO

from accessibility import check_endpoint
from summarize import * #json_keys, data_frame

### Interoperability Assessment of EuroArgo JSON api (Floats metadata)

**Endpoint documentation**

API documentation (swagger): https://fleetmonitoring.euro-argo.eu/swagger-ui.html

### Endpoint

In [2]:
base = "https://fleetmonitoring.euro-argo.eu"

##### Table Of Content

- [Exploring the JSON API](#exploring-the-json-api)
- [Technical interoperability](#technical-interoperability)
    - [Standards compliance](#standards-compliance) 
    - [Interface consistency](#interface-consistency) 
    - [Versioning & Backward compatibility](#versioning-and-backwardscompatibility) 

- [Semantic interoperability](#semantic-interoperability)
    - [Documentation](#documentation) 
    - [Metadata](#metadata)
    - [Shared Vocabulary & Ontologies](#shared-vocabularies-and-ontologies)
    - [Contextual Meaning](#contextual-meaning)
- ([note on Data Granularity](#note-on-data-granularity))
     

### Exploring the JSON API

Exploration of the API was done through the swagger documentation and for GET requests also below in this notebook:

**Floats** (get all floats basic data)

In [ ]:
float_url = "https://fleetmonitoring.euro-argo.eu/floats"

float_md = requests.get(float_url).json()
summary = json_keys(float_md)
float_md_summ = pd.DataFrame([
    {
        "Property": key,
        "Count": info["count"],
        "Types": ', '.join(info["types"]),
        "Example": info["example"]
    }
    for key, info in summary.items()
])
float_md_summ #.to_csv("properties/ARGO_JSONAPI_metadata_floats_2.csv", index=False)

This endpoint returns a.o. metadata for a specific float. The data model can be visualized as:  
![image.png](images/ARGO_JSONAPI_float.drawio.png)

**Export** (get export metadata)

In [ ]:
float_url = "https://fleetmonitoring.euro-argo.eu/export?wmo=7902313"

float_md = requests.get(float_url).text

float_md_df = pd.read_csv(StringIO(float_md), sep=";")
float_md_df

# This appears to return metadata relating to the cycles performed by a platform 

# (note: units captured in column title)

**Statistics** (get functional monitoring statistics data)

In [ ]:
stats_url = "https://fleetmonitoring.euro-argo.eu/functionalMonitoring/statistics"

stats_md = requests.get(stats_url).json()

summary = json_keys(stats_md)

stats_md_summ = pd.DataFrame([
    {
        "Property": key,
        "Count": info["count"],
        "Types": ', '.join(info["types"]),
        "Example": info["example"]
    }
    for key, info in summary.items()
])
stats_md_summ

This endpoint returns functional monitoring stats for the API. The properties can be visualized as:  
![image.png](images/ARGO_JSONAPI_stats.drawio.png)

**Get creation data** (get metadata index creation date)

In [ ]:
index_url = "https://fleetmonitoring.euro-argo.eu/metadataIndex/get-creation-date"

index_md = requests.get(index_url).text
index_md

**Platform codes** (get all platform codes)

In [ ]:
platfcodes_url = "https://fleetmonitoring.euro-argo.eu/platformCodes"

platfcodes_md = requests.get(platfcodes_url).json()
platfcodes_md

# Returns a list of (assumed to be wmo?) platform codes

**Technical data** (get a float's technical data with values)

In [ ]:
technical_url = "https://fleetmonitoring.euro-argo.eu/technical-data/13857"

technical_md = requests.get(technical_url).json()

summary = json_keys(technical_md)


technical_md_summ = pd.DataFrame([
    {
        "Property": key,
        "Count": info["count"],
        "Types": ', '.join(info["types"]),
        "Example": info["example"]
    }
    for key, info in summary.items()
])
technical_md_summ

This endpoint returns technical metadata for the specified float/platform/wmo. The properties can be visualized as:  
![image.png](images/ARGO_JSONAPI_technical.drawio.png)

**Version** (get the API server version)

In [ ]:
version_url = "https://fleetmonitoring.euro-argo.eu/get-version"

version_md = requests.get(version_url).text
version_md


### Technical interoperability

#### Standards compliance 
*Involves checking compliance to HTTP standard, JSON schema, JSON API spec*

In [4]:
# various endpoints
urls = [
    'https://fleetmonitoring.euro-argo.eu/',                                           #base
    'https://fleetmonitoring.euro-argo.eu/floats',                                     #example endpoint from documentation
    'https://fleetmonitoring.euro-argo.eu/functionalMonitoring/statistics',            #example endpoint from documentation 
    'https://fleetmonitoring.euro-argo.eu/platformCodes',                              #example endpoint from documentation
    'https://fleetmonitoring.euro-argo.eu/technical-data/13857',                       #example endpoint from documentation, with specified input parameters 
    'https://fleetmonitoring.euro-argo.eu/technical-data/'                             #faulty endpoint

]

# headers of interest
important_headers = [
    "Content-Type",
    "Content-Length",
    "Content-Encoding",
    "Transfer-Encoding",
    "Vary",
    "Accept",
    "Accept-Encoding",
    "Accept-Language",
    "Server",
    "Date",
    "Strict-Transport-Security"
]

for url in urls:
    print("="*100)
    print(f"🔗 URL: {url}")
    try:
        r = requests.get(url, timeout=10)
        print(f"Status code: {r.status_code}\n")

        # filter only important headers
        headers_filtered = {k: v for k, v in r.headers.items() if k in important_headers}
        
        print("📌 Relevant headers:")
        if headers_filtered:
            for k, v in headers_filtered.items():
                print(f"   {k}: {v}")
        else:
            print("   (none of the selected headers present)")
        
        # try parsing response
        print("\n📌 Response preview:")
        try:
            data = r.json()
            if isinstance(data, dict):
                print("   JSON object with keys:", list(data.keys()))
            elif isinstance(data, list):
                print(f"   JSON list with {len(data)} items")
            else:
                print("   JSON response (other type)")
        except ValueError:
            print("   Text response (first 300 chars):")
            print("   " + r.text[:300].replace("\n", " ") + " ...")
    
    except requests.exceptions.RequestException as e:
        print(f"❌ Request failed: {e}")

🔗 URL: https://fleetmonitoring.euro-argo.eu/
Status code: 200

📌 Relevant headers:
   Date: Wed, 27 Aug 2025 12:34:05 GMT
   Server: Apache
   Content-Type: text/html;charset=ISO-8859-1
   Vary: Accept-Encoding
   Content-Encoding: gzip
   Content-Length: 7431

📌 Response preview:
   Text response (first 300 chars):
   <!doctype html> <html lang="en" data-critters-container> <head>   <meta charset="utf-8">   <title>Argo Fleet Monitoring - Euro-Argo</title>   <base href="/">    <meta name="viewport" content="width=device-width, initial-scale=1">   <meta name="robots" content="index,nofollow">   <meta name="descript ...
🔗 URL: https://fleetmonitoring.euro-argo.eu/floats
❌ Request failed: HTTPSConnectionPool(host='fleetmonitoring.euro-argo.eu', port=443): Read timed out. (read timeout=10)
🔗 URL: https://fleetmonitoring.euro-argo.eu/functionalMonitoring/statistics
Status code: 200

📌 Relevant headers:
   Date: Wed, 27 Aug 2025 12:34:15 GMT
   Server: Apache
   Content-Type: application/jso

#### Interface consistency
*Involves checking naming conventions, field types and data formats, error messages, cross-platform support (language neutral data representation, no additional data transformations needed when consuming the data)*

In [6]:
import requests
import json

def get_openapi_spec(base_url):
    for path in ["/v2/api-docs", "/v3/api-docs", "/openapi.json", "/swagger.json"]:
        url = base_url.rstrip("/") + path
        r = requests.get(url)
        if r.ok and "json" in r.headers.get("content-type", ""):
            return r.json()
    return None

# Example: Euro-Argo Fleet Monitoring API
spec = get_openapi_spec(base)
print(json.dumps(spec, indent=2))

endpoint = "/api/platforms"   # choose the endpoint you want to inspect
if spec and endpoint in spec["paths"]:
    details = spec["paths"][endpoint]
    print(json.dumps(details, indent=2))
else:
    print(f"Endpoint {endpoint} not found in spec")

{
  "swagger": "2.0",
  "info": {
    "description": "Api Documentation",
    "version": "1.0",
    "title": "Api Documentation",
    "termsOfService": "urn:tos",
    "contact": {},
    "license": {
      "name": "Apache 2.0",
      "url": "http://www.apache.org/licenses/LICENSE-2.0"
    }
  },
  "host": "fleetmonitoring.euro-argo.eu",
  "basePath": "/",
  "tags": [
    {
      "name": "metadata-index-date-controller",
      "description": "Metadata Index Date Controller"
    },
    {
      "name": "contact-controller",
      "description": "Contact Controller"
    },
    {
      "name": "functional-monitoring-controller",
      "description": "Functional Monitoring Controller"
    },
    {
      "name": "export-controller",
      "description": "Export Controller"
    },
    {
      "name": "autonomous-float-controller",
      "description": "Autonomous Float Controller"
    },
    {
      "name": "caching-controller",
      "description": "Caching Controller"
    },
    {
      "name

#### Versioning & Backward compatibility
*Involves checking whether the API has a clear versioning strategy and maintains support for older clients when changes to the schema occur*

In [1]:
import requests

# Try typical OpenAPI/Swagger spec URLs
urls = [
    "https://fleetmonitoring.euro-argo.eu/v3/api-docs",
    "https://fleetmonitoring.euro-argo.eu/v2/api-docs",
    "https://fleetmonitoring.euro-argo.eu/openapi.json",
]

for url in urls:
    try:
        r = requests.get(url, timeout=10)
        if r.ok and r.headers.get("content-type", "").startswith("application/json"):
            data = r.json()
            version = data.get("info", {}).get("version")
            print(f"{url} --> version: {version}")
    except Exception as e:
        print(f"{url} --> failed ({e})")

https://fleetmonitoring.euro-argo.eu/v2/api-docs --> version: 1.0


Findings:  

**Protocol Compliance**  
- HTTP protocol
    - base url doesn't return json response
    - wrongly formatted endpoint (./technical-data/) correctly responds with 404 status code and also the content-type is correct

- JSON schema
    - No schemas are defined or available through the API documentation.
    - As a result, adherence of endpoint responses to schemas could not be tested.

**Interface Consistency**  
- Use of camelCase naming convention.
- Correct error messages are provided for wrongly formatted endpoints.

**Versioning & Backwards compatibility**  
- Backward compatibility between API versions is an important aspect of technical interoperability.
- However, since only the latest version (V3) is accessible and other versions return a 404, verification was not possible.

### Semantic interoperability

#### Documentation
*Involves checking availablity and content of swagger documentation (e.g. definitions include descriptions not just types)*

API documentation available at https://fleetmonitoring.euro-argo.eu/swagger-ui.html

The Swagger API documentation clearly lists the available endpoints for a specific HTTP method.  
The listed endpoints can be dynamically explored and tested (with or without input parameters).  

The schemas definitions are not seperately included.  

In the dynamical exploration of the endpoints, the definitions of input parameters are not given, nor any example values (this is understandable in post requests not in get requests); only datatypes are given.  
Some endpoints require knowledge on wmo-codes, platform-codes, ...

![example-response-body](./images/EuroArgo_JsonAPI_metadata_exampleresponsebody.png)



#### Metadata 
*Involves checking use of self-describing/unambiguous terms, measurements of units defined, ...*

Example response body of get request for a cycle trajectory by platformId indicates:

- Terms are self-describing to some level (e.g. `cycleId` and `cycleNumber` ~ the identifer and name of a cycle),  
however due to lack of schema definitions and descriptions, terms are ambiguous (e.g. what is a cycle?) 

- Measurements of units are not defined  
(e.g. pres, psal, temp values are represented by integer, the unit of measurement is not captured here, nor in antoher key in the response body)  

![example-response-body](./images/EuroArgo_JsonAPI_metadata_exampleresponsebody2.png)

#### Shared Vocabulary & Ontologies
*Involves checking use of terms from standard vocabularies (ISO, schema.org, ...) and, if applicable, alignment with domain ontologies*

- Dates formats are following ISO 8601 standard.

- Limited/no explicit adherence to other standards (like schema.org, DCAT) or domain standards. This is not mentioned in documentation.   

- Limited / no use of persistent identifiers (URIs) - e.g. wmo-codes/platform-codes, types, ... could be represented by URLs from existing standards. 

#### Contextual Meaning
*Involves checking context terms (e.g. `"status":"active"`) and enumerations are clearly defined so their meaning is consistent across different systems*

- Occurrence of 'status' context term `"status":"active"`, which is only unambiguously defined through the nested json structure


**Note on data granularity**

to include!